# NB8 — Mitochondrial QC Threshold Sensitivity

Sensitivity analysis for the 20% mitochondrial read fraction threshold applied during
quality control. Re-filters each microglia cohort at four thresholds (MT% <= 20%, 25%,
30%, 35%) and recomputes donor-level (or cell-level) MES-tolerance-positioning Spearman
correlations. Generates Supplementary Table S24.

Addresses **Reviewer 3, Comment 16**: Fixed QC thresholds may be overly stringent in
neuroinflammatory settings, where increased mitochondrial transcription can reflect
genuine inflammatory or metabolic activation rather than poor-quality cells.

**Paper:** Zafar SA, Qin W, Liu C, Khan AA, Faisal MS.
*Thymus-derived myeloid programs track microglial tolerance states across human cohorts.* (2026).

**Depends on:** NB4/NB5 scored `.h5ad` files (must contain `pct_counts_mt` in `.obs`,
or raw counts so it can be recomputed, and `MES0X_score` columns).

**Outputs:**
- `Supplementary_TableS24_Mito_Threshold_Sensitivity.xlsx` (sheets: `long`, `pivot`)
- `Supplementary_FigS24_Mito_Threshold_Sensitivity.png`

> **Note:** Set `MES_BASE_DIR` to your project root before running.
> Raw data can be obtained from the public repositories listed in Supplementary Table S1.

In [ ]:
from __future__ import annotations

import os, re, time, gc, warnings, traceback
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import scipy.stats as scipy_stats
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         8,
    "axes.titlesize":    9,
    "axes.labelsize":    8,
    "xtick.labelsize":   7,
    "ytick.labelsize":   7,
    "legend.fontsize":   7,
    "figure.dpi":        150,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.05,
    "axes.linewidth":    0.8,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size":  3,
    "ytick.major.size":  3,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_style('ticks')
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    print('[WARN] seaborn not installed; figures will use matplotlib only.')

import anndata as ad
import scanpy as sc

try:
    from statsmodels.stats.multitest import multipletests
    HAS_SM = True
except ImportError:
    HAS_SM = False
    print('[WARN] statsmodels not installed; BH correction will use scipy fallback.')


# =============================================================================
# 0) PATHS
# =============================================================================

np.random.seed(42)

BASE_DIR  = Path(os.environ.get("MES_BASE_DIR", "."))  # <-- SET PATH
PROC_DIR  = BASE_DIR / "Process Data" / "aim2_microglia"
MANUS_DIR = BASE_DIR / "Manuscript data"
TAB_DIR   = MANUS_DIR / "Tables"
FIG_DIR   = MANUS_DIR / "Figures"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SCORED_GLOB = str(PROC_DIR / "*__microglia_scored.h5ad")

DPI              = 1200
N_BOOT           = 300
ALPHA            = 0.05
MIN_CELLS_KEEP   = 100

MITO_THRESHOLDS  = [20, 25, 30, 35]
MES_COLS         = [f"MES0{i}" for i in range(1, 9)]


# =============================================================================
# 1) LOGGING AND I/O HELPERS
# =============================================================================

def _now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str) -> None:
    print(f"[{_now()}] {msg}", flush=True)

def save_fig(path: Path, fig=None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    (fig or plt).savefig(path, dpi=DPI, bbox_inches="tight")
    if fig is not None:
        plt.close(fig)
    else:
        plt.close()
    log(f"SAVED FIG  {path}")

def save_xlsx_multi(dfs: Dict[str, pd.DataFrame], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        for name, df in dfs.items():
            if df is not None and not (isinstance(df, pd.DataFrame) and df.empty):
                df.to_excel(w, index=False, sheet_name=name[:31])
    log(f"SAVED TABLE {path}")


# =============================================================================
# 2) STATISTICAL HELPERS
# =============================================================================

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    if HAS_SM:
        _, q, _, _ = multipletests(pvals, method="fdr_bh")
        return q
    n = len(pvals)
    order = np.argsort(pvals)
    q = np.empty(n)
    q[order] = pvals[order] * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    return np.minimum(q, 1.0)


def safe_corr_with_ci(
    x: pd.Series,
    y: pd.Series,
    method: str = "spearman",
    n_boot: int = N_BOOT,
    seed: int = 42,
) -> Dict:
    x = pd.to_numeric(x, errors="coerce").to_numpy()
    y = pd.to_numeric(y, errors="coerce").to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 5:
        return {"r": np.nan, "ci_lo": np.nan, "ci_hi": np.nan, "p": np.nan, "n": n}
    if method == "spearman":
        r_obs, p_obs = scipy_stats.spearmanr(x, y)
        def _corr(a, b): return scipy_stats.spearmanr(a, b)[0]
    else:
        r_obs, p_obs = scipy_stats.pearsonr(x, y)
        def _corr(a, b): return scipy_stats.pearsonr(a, b)[0]
    rng = np.random.default_rng(seed)
    boots = np.array([
        _corr(x[idx := rng.integers(0, n, n)], y[idx])
        for _ in range(n_boot)
    ])
    boots = boots[np.isfinite(boots)]
    ci_lo = float(np.percentile(boots, 2.5))  if len(boots) else np.nan
    ci_hi = float(np.percentile(boots, 97.5)) if len(boots) else np.nan
    return {"r": float(r_obs), "ci_lo": ci_lo, "ci_hi": ci_hi, "p": float(p_obs), "n": n}


# =============================================================================
# 3) COHORT HELPERS
# =============================================================================

def detect_donor_col(obs: pd.DataFrame) -> Optional[str]:
    candidates = [
        "donor_id", "donor", "subject", "SubjectID", "subject_id",
        "individualID", "individual_id", "case_id", "sample",
        "Participant ID", "participant_id",
    ]
    for c in candidates:
        if c in obs.columns:
            n_unique = obs[c].nunique()
            if 2 <= n_unique <= 500:
                return c
    return None


def donor_aggregate(
    obs: pd.DataFrame,
    donor_col: str,
    score_cols: List[str],
) -> pd.DataFrame:
    cols = [c for c in score_cols if c in obs.columns]
    return obs.groupby(donor_col)[cols].mean().reset_index()


def ensure_pct_mt(adata: ad.AnnData) -> bool:
    # Ensure pct_counts_mt is in adata.obs; compute if missing. Returns True on success.
    if "pct_counts_mt" in adata.obs.columns:
        return True
    try:
        adata.var["mt"] = adata.var_names.astype(str).str.upper().str.startswith("MT-")
        sc.pp.calculate_qc_metrics(
            adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
        )
        return "pct_counts_mt" in adata.obs.columns
    except Exception as e:
        log(f"  Could not compute pct_counts_mt: {e}")
        return False


TOL_HOMEO = ["P2RY12", "CX3CR1", "TMEM119", "GPR34", "SALL1", "CSF1R", "OLFML3"]
TOL_ACTIV = ["APOE", "SPP1", "LPL", "TREM2", "CST7", "CTSD", "TYROBP",
             "FCER1G", "LGALS3", "CD68"]


def ensure_tolerance(adata: ad.AnnData) -> None:
    if "tolerance_positioning" not in adata.obs.columns:
        var_upper = adata.var_names.astype(str).str.upper()
        upper_to_orig = dict(zip(var_upper, adata.var_names))
        def score(genes, key):
            present = [upper_to_orig[g] for g in genes if g in var_upper]
            if len(present) >= 3:
                sc.tl.score_genes(adata, present, score_name=key, use_raw=False)
        score(TOL_HOMEO, "_tol_h8")
        score(TOL_ACTIV, "_tol_a8")
        if "_tol_h8" in adata.obs and "_tol_a8" in adata.obs:
            adata.obs["tolerance_positioning"] = (
                adata.obs["_tol_h8"] - adata.obs["_tol_a8"]
            )


# =============================================================================
# 4) LOAD SCORED MICROGLIA COHORTS
# =============================================================================

log("=" * 72)
log("NB8: Mitochondrial QC Threshold Sensitivity")
log("=" * 72)

import glob as _glob

log("Loading scored microglia cohorts ...")
h5ad_files = sorted(_glob.glob(SCORED_GLOB))
assert len(h5ad_files) >= 4, (
    f"Expected >=4 scored .h5ad files in {PROC_DIR}, found {len(h5ad_files)}. "
    "Re-run NB4 and NB5 first."
)

cohorts: Dict[str, ad.AnnData] = {}
for fp in h5ad_files:
    name = Path(fp).stem.replace("__microglia_scored", "")
    log(f"  Reading {name} ...")
    cohorts[name] = sc.read_h5ad(fp)
    gc.collect()

log(f"Loaded {len(cohorts)} cohorts: {list(cohorts.keys())}")


# =============================================================================
# 5) A11 - MITOCHONDRIAL THRESHOLD SENSITIVITY
# =============================================================================

log("=" * 72)
log(f"A11: MT% threshold sensitivity ({MITO_THRESHOLDS})")
log("=" * 72)

try:
    a11_rows = []

    for ds, adata in cohorts.items():
        log(f"  Processing {ds} ...")

        if not ensure_pct_mt(adata):
            log(f"    SKIP {ds}: pct_counts_mt unavailable")
            continue

        ensure_tolerance(adata)
        if "tolerance_positioning" not in adata.obs.columns:
            log(f"    SKIP {ds}: tolerance_positioning not available")
            continue

        mes_score_cols = [f"{m}_score" for m in MES_COLS]
        mes_score_cols = [c for c in mes_score_cols if c in adata.obs.columns]
        if not mes_score_cols:
            log(f"    SKIP {ds}: no MES score columns found")
            continue

        donor_col = detect_donor_col(adata.obs)
        mt_pct    = pd.to_numeric(adata.obs["pct_counts_mt"], errors="coerce").to_numpy()

        for thr in MITO_THRESHOLDS:
            keep   = (mt_pct <= thr) & np.isfinite(mt_pct)
            n_kept = int(keep.sum())

            if n_kept < MIN_CELLS_KEEP:
                log(f"    SKIP {ds} @ MT<={thr}%: only {n_kept} cells retained")
                continue

            sub_obs    = adata.obs.loc[keep].copy()
            score_cols = ["tolerance_positioning"] + mes_score_cols
            score_cols = [c for c in score_cols if c in sub_obs.columns]

            if donor_col is not None and donor_col in sub_obs.columns:
                ddf   = donor_aggregate(sub_obs, donor_col, score_cols)
                level = "donor"
            else:
                ddf   = sub_obs[score_cols].copy()
                level = "cell"

            for mes_sc in mes_score_cols:
                mes_name = mes_sc.replace("_score", "")
                if mes_sc not in ddf.columns:
                    continue
                r = safe_corr_with_ci(
                    ddf["tolerance_positioning"],
                    ddf[mes_sc],
                    method="spearman",
                    n_boot=N_BOOT,
                )
                a11_rows.append({
                    "dataset":         ds,
                    "mito_thresh_pct": thr,
                    "level":           level,
                    "n_cells_kept":    n_kept,
                    "N_unit":          r["n"],
                    "MES":             mes_name,
                    "r":               r["r"],
                    "p":               r["p"],
                    "ci_lo":           r["ci_lo"],
                    "ci_hi":           r["ci_hi"],
                })

        gc.collect()

    df_a11 = pd.DataFrame(a11_rows)
    assert not df_a11.empty, "A11 produced no rows - check cohort loading and MT column."

    df_a11["q_BH"] = bh_fdr(df_a11["p"].fillna(1.0).values)

    pivot = df_a11.pivot_table(
        index=["dataset", "MES"],
        columns="mito_thresh_pct",
        values="r",
        aggfunc="mean",
    ).reset_index()

    for thr in MITO_THRESHOLDS:
        if thr in pivot.columns:
            pivot.rename(columns={thr: f"rho_mt{thr}pct"}, inplace=True)

    ref_col = "rho_mt20pct"
    for thr in [25, 30, 35]:
        cmp_col = f"rho_mt{thr}pct"
        if ref_col in pivot.columns and cmp_col in pivot.columns:
            pivot[f"delta_vs_20pct_{thr}"] = pivot[cmp_col] - pivot[ref_col]

    delta_cols = [c for c in pivot.columns if c.startswith('delta_')]
    if delta_cols:
        all_delta = pivot[delta_cols].abs().values.ravel()
        all_delta = all_delta[np.isfinite(all_delta)]
        log(f"  Max    |Delta_rho| across all thresholds: {np.max(all_delta):.6f}")
        log(f"  Median |Delta_rho| across all thresholds: {np.median(all_delta):.6f}")
        if "delta_vs_20pct_35" in pivot.columns:
            med_35 = pivot["delta_vs_20pct_35"].abs().median()
            log(f"  Median |Delta_rho| 35% vs 20% (manuscript claim <0.001): {med_35:.6f}")

    out_table = TAB_DIR / "Supplementary_TableS24_Mito_Threshold_Sensitivity.xlsx"
    save_xlsx_multi({"long": df_a11, "pivot": pivot}, out_table)

except Exception as e:
    log(f"A11 FAILED: {e}\n{traceback.format_exc()}")
    df_a11, pivot = pd.DataFrame(), pd.DataFrame()


# =============================================================================
# 6) FIGURE
# =============================================================================

log("Generating sensitivity figure ...")

try:
    assert not df_a11.empty

    datasets = df_a11["dataset"].unique().tolist()
    n_ds  = len(datasets)
    n_col = min(n_ds, 4)
    n_row = int(np.ceil(n_ds / n_col))

    fig, axes = plt.subplots(
        n_row, n_col,
        figsize=(n_col * 3.5, n_row * 3.2),
        squeeze=False,
    )

    CMAP = "RdBu_r"
    VABS = 0.8

    for idx, ds in enumerate(datasets):
        ax  = axes[idx // n_col][idx % n_col]
        sub = df_a11[df_a11["dataset"] == ds].copy()
        mat = sub.pivot(index="MES", columns="mito_thresh_pct", values="r")
        mes_order = [m for m in MES_COLS if m in mat.index]
        mat = mat.reindex(index=mes_order)
        mat.columns = [f"MT<={c}%" for c in mat.columns]

        im = ax.imshow(
            mat.values.astype(float),
            cmap=CMAP, vmin=-VABS, vmax=VABS, aspect='auto',
        )
        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels(mat.columns, rotation=0, fontsize=7)
        ax.set_yticks(range(len(mat.index)))
        ax.set_yticklabels(mat.index, fontsize=7)
        ax.set_title(ds.replace("_", " "), fontsize=8, pad=4)

        for r_i in range(mat.shape[0]):
            for c_i in range(mat.shape[1]):
                val = mat.values[r_i, c_i]
                if np.isfinite(val):
                    ax.text(
                        c_i, r_i, f'{val:.3f}',
                        ha='center', va='center', fontsize=6,
                        color='white' if abs(val) > 0.4 else 'black',
                    )

        plt.colorbar(im, ax=ax, shrink=0.7, label="Spearman rho", pad=0.02)

    for idx in range(n_ds, n_row * n_col):
        axes[idx // n_col][idx % n_col].set_visible(False)

    fig.suptitle(
        "MES-Tolerance Correlation: Mitochondrial QC Threshold Sensitivity",
        fontsize=9, y=1.01,
    )
    plt.tight_layout()
    save_fig(FIG_DIR / "Supplementary_FigS24_Mito_Threshold_Sensitivity.png", fig)

except Exception as e:
    log(f"FIGURE FAILED: {e}\n{traceback.format_exc()}")


# =============================================================================
# 7) COMPLETION REPORT
# =============================================================================

log("=" * 72)
log("NB8 COMPLETED")
log("=" * 72)
log(f"Tables  : {TAB_DIR}")
log(f"Figures : {FIG_DIR}")
log("")
log("Key outputs:")
log("  Supplementary_TableS24_Mito_Threshold_Sensitivity.xlsx")
log("    Sheet long   - per (dataset, MES, threshold): rho, CI, p, q_BH, n_cells_kept")
log("    Sheet pivot  - per (dataset, MES): rho at MT<=20/25/30/35% + delta vs 20%")
log("  Supplementary_FigS24_Mito_Threshold_Sensitivity.png")
log("")
log("STATISTICAL METHODS:")
log("  Spearman rank correlation at donor level (cell level where donor metadata unavailable)")
log("  95% CI: 300-iteration bootstrap resampling")
log("  Multiple testing: BH-FDR across all tests at each threshold")
log(f"  Thresholds tested: {MITO_THRESHOLDS} % mitochondrial reads")
log("  Delta columns show change relative to default 20% threshold")
